## Simple mjSpec tutorial
- Use mjSpec to synthesize mujoco scene

In [1]:
import mujoco as mj
import mujoco_viewer

import numpy as np
import time

import xml.etree.ElementTree as ET
from lxml import etree

In [2]:
def print_xml(xml_input,color=True):
    if isinstance(xml_input, ET.Element):
        rough_string = ET.tostring(xml_input, encoding='unicode')
    else:
        rough_string = xml_input

    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.fromstring(rough_string, parser=parser)
    pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
    print(pretty_xml)

### Basic example - check xml

In [3]:
# simple spec

spec = mj.MjSpec()
print(spec) # mj spec object
print_xml(spec.to_xml())

<mujoco model="MuJoCo Model">
  <compiler angle="radian"/>
  <worldbody/>
</mujoco>



- default model name = MuJoCo Model
- default compiler angle = radian

In [4]:
arena_xml = """
<mujoco>
<visual>
    <headlight diffuse=".5 .5 .5" specular="1 1 1"/>
    <global offwidth="2048" offheight="1536"/>
    <quality shadowsize="8192"/>
</visual>

<asset>
    <texture type="skybox" builtin="gradient" rgb1="1 1 1" rgb2="1 1 1" width="10" height="10"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="1 1 1" rgb2="1 1 1" markrgb="0 0 0" width="400" height="400"/>
    <material name="groundplane" texture="groundplane" texrepeat="45 45" reflectance="0"/>
</asset>

<worldbody>
    <geom name="floor" size="150 150 0.1" type="plane" material="groundplane"/>
</worldbody>
</mujoco>
"""

spec = mj.MjSpec.from_string(arena_xml)
print_xml(spec.to_xml())

<mujoco model="MuJoCo Model">
  <compiler angle="radian"/>
  <visual>
    <global offwidth="2048" offheight="1536"/>
    <quality shadowsize="8192"/>
    <headlight diffuse="0.5 0.5 0.5" specular="1 1 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="1 1 1" rgb2="1 1 1" width="10" height="60"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="1 1 1" rgb2="1 1 1" width="400" height="400"/>
    <material name="groundplane" texture="groundplane" texrepeat="45 45"/>
  </asset>
  <worldbody>
    <geom name="floor" size="150 150 0.1" type="plane" material="groundplane"/>
  </worldbody>
</mujoco>



### Adding objects: body, geom, site

In [5]:
path = '../floor_white_gray.xml'
spec = mj.MjSpec.from_file(path)

In [6]:
# add specific body to worldbody
body = spec.worldbody.add_body(
    name="base_link",
    pos = np.zeros((3)),
    euler = [0, 0.8, 0] # auto-fixed to quat
)
print_xml(spec.to_xml())


<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0"/>
  </worldbody>
</mujoco>



In [7]:
print(body) # returns spec body object
print(body == spec.body("base_link"))

True


In [8]:
# add geom to body
geom = body.add_geom(
    name="base_link_geom",
    pos = [0.0, 0.0, 0.1],
    type=mj.mjtGeom.mjGEOM_BOX,
    size = [0.1, 0.1, 0.1],
    rgba=[0,1,0,1]
)
spec.body("base_link").add_site(
    name="base_link_site"
)

print(geom)

In [9]:
print(body)
print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" rgba="0 1 0 1"/>
      <site name="base_link_site" pos="0 0 0"/>

In [10]:
# add hierarchical body -> under body 1
body2 = body.add_body(
    name="link1",
    pos = [0.0,0.0,0.3],
)

body2.add_geom(
    name="link1_geom",
    pos = [0.0,0.0,0.1],
    type=mj.mjtGeom.mjGEOM_SPHERE,
    size = [0.1, 0.0, 0.0],
    rgba=[1,0,0,1]
)

In [11]:
print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" rgba="0 1 0 1"/>
      <site name="base_link_site" pos="0 0 0"/>

In [12]:
frame1 = spec.worldbody.add_frame(pos=[0,3,3], quat=[0, 0, 0, 1])

# not represented in xml

# frame doesn't have add_body
# frame1.add_body(
#     name="body_frame_1",
#     pos = [0.0,0.0,0.3],
# )


# instead, use attach!
# attach body3 to frame
# body already included in spec cannot be attached to spec
arena_xml = """
<mujoco>
<worldbody>
    <body name="box" pos="0 0 0">
        <geom type="box" size="1 1 1"/>
    </body>
</worldbody>
</mujoco>
"""

additional_spec = mj.MjSpec.from_string(arena_xml)
body3 = additional_spec.body('box')

body4 = frame1.attach_body(body3, 'attached-', '-1') # body object, prefix, postfix

print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <default>
    <default class="attached-main-1"/>
  </default>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" 

In [13]:
model = spec.compile()
data = mj.MjData(model)

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# data reset
mj.mj_resetData(model, data)

while True:
    if viewer.is_alive:

        # apply control
        # apply_control_name(model, data, name=["shoulder_lift", "elbow"], value=[0.8, 0.8]) # now data 
        # control_signal = [0.] * 6
        
        mj.mj_step(model, data)
        viewer.render()

    else:
        break

# close
viewer.close()

In [ ]:
# attachment_frame.attach_body(spec_humanoid.body('torso'), 'a', 'b')
# spec_robot = mj.MjSpec.from_file(path)

NameError: name 'attachment_frame' is not defined